# Custom CNN — complete training notebook

This notebook contains the complete data preparation, model construction, training, evaluation and export workflow. No training module outside this notebook is used. Run cells in order only after reviewing them.

**Goal:** learn increasingly complex X-ray features with a compact convolutional network trained locally from random initialization. The final cell registers the artifact so the backend and model selector can use it.


## End-to-end flow

```text
image files → parallel decode/resize → augmentation → convolution blocks
            → global feature pooling → pneumonia probability
            → untouched test metrics → .keras artifact → web application
```

The test set is isolated until the final evaluation. This prevents test information leaking into model selection.


## 1. Paths and experiment controls

These values make the run reproducible and easy to tune. A smaller image size and batch of 32 keep this baseline practical on CPU while preserving enough lung detail for a first model.


In [1]:
from pathlib import Path
from datetime import datetime, timezone
import hashlib
import json
import os
import random

import numpy as np
import tensorflow as tf

ROOT = Path.cwd().resolve().parent if Path.cwd().name == "notebooks" else Path.cwd().resolve()
DATASET = ROOT / "dataset" / "archive" / "chest_xray"
CATALOG = ROOT / "server" / "model_catalog.json"
ARTIFACT = ROOT / "server" / "artifacts" / "custom-cnn.keras"
MODEL_ID = "custom-cnn"
IMAGE_SIZE = 160
BATCH_SIZE = 32
EPOCHS = 30
LEARNING_RATE = 0.001
SEED = 42

assert DATASET.is_dir(), f"Dataset not found: {DATASET}"
ARTIFACT.parent.mkdir(parents=True, exist_ok=True)


## 2. CPU/GPU execution and multithreading

TensorFlow already parallelizes numerical kernels. The explicit thread pools use the available CPU cores, while `AUTOTUNE` below overlaps file decoding and preprocessing with model execution. Training several models at once would usually slow them down through RAM/GPU contention, so each notebook trains one model efficiently.


In [2]:
# Configure TensorFlow before it creates its execution context.
CPU_THREADS = max(1, (os.cpu_count() or 2) - 1)
try:
    tf.config.threading.set_intra_op_parallelism_threads(CPU_THREADS)
    tf.config.threading.set_inter_op_parallelism_threads(max(1, CPU_THREADS // 2))
except RuntimeError:
    print("Thread pools already initialized; restart the kernel to change them.")

random.seed(SEED)
np.random.seed(SEED)
tf.keras.utils.set_random_seed(SEED)
print({"cpu_threads": CPU_THREADS, "gpus": tf.config.list_physical_devices("GPU")})


{'cpu_threads': 19, 'gpus': []}


## 3. Discover and split the data

The training and small validation folders are combined, then split 80/20 within each class. Stratification preserves the Normal/Pneumonia ratio. Seed 42 guarantees the same samples are assigned on every run. The original test folder is never used here.


In [3]:
EXTENSIONS = {".jpg", ".jpeg", ".png"}

def image_files(split, class_name, label):
    folder = DATASET / split / class_name
    if not folder.is_dir():
        raise FileNotFoundError(f"Missing dataset folder: {folder}")
    return [(str(path), label) for path in sorted(folder.iterdir()) if path.suffix.lower() in EXTENSIONS]

pool = image_files("train", "NORMAL", 0) + image_files("train", "PNEUMONIA", 1)
pool += image_files("val", "NORMAL", 0) + image_files("val", "PNEUMONIA", 1)
test_samples = image_files("test", "NORMAL", 0) + image_files("test", "PNEUMONIA", 1)

rng = random.Random(SEED)
train_samples, validation_samples = [], []
for label in (0, 1):
    group = [sample for sample in pool if sample[1] == label]
    rng.shuffle(group)
    cut = round(len(group) * 0.2)
    validation_samples.extend(group[:cut])
    train_samples.extend(group[cut:])
rng.shuffle(train_samples)
rng.shuffle(validation_samples)

def counts(samples):
    return {"normal": sum(label == 0 for _, label in samples), "pneumonia": sum(label == 1 for _, label in samples)}

{"train": counts(train_samples), "validation": counts(validation_samples), "test": counts(test_samples)}


{'train': {'normal': 1079, 'pneumonia': 3106},
 'validation': {'normal': 270, 'pneumonia': 777},
 'test': {'normal': 234, 'pneumonia': 390}}

## 4. Build the parallel input pipeline

Image decoding and resizing run in parallel. Prefetch prepares the next batch while the current batch trains. Class weights counter the dataset imbalance so missing Normal cases is penalized rather than allowing the majority class to dominate accuracy.


In [4]:
def load_image(path, label):
    image = tf.io.decode_image(tf.io.read_file(path), channels=3, expand_animations=False)
    image.set_shape((None, None, 3))
    image = tf.image.resize(tf.cast(image, tf.float32), (IMAGE_SIZE, IMAGE_SIZE)) / 255.0
    return image, tf.cast(label, tf.float32)

def make_dataset(samples, training=False):
    paths, labels = zip(*samples)
    data = tf.data.Dataset.from_tensor_slices((list(paths), list(labels)))
    if training:
        data = data.shuffle(len(samples), seed=SEED, reshuffle_each_iteration=True)
    return (data
            .map(load_image, num_parallel_calls=tf.data.AUTOTUNE)
            .batch(BATCH_SIZE)
            .prefetch(tf.data.AUTOTUNE))

train_data = make_dataset(train_samples, training=True)
validation_data = make_dataset(validation_samples)
test_data = make_dataset(test_samples)

class_counts = np.bincount([label for _, label in train_samples], minlength=2)
class_weight = {index: len(train_samples) / (2 * count) for index, count in enumerate(class_counts)}
class_weight


{0: np.float64(1.9392956441149212), 1: np.float64(0.6736960721184804)}

## 5. Define the model from random initialization

Each block first learns local patterns, normalizes activations for stable optimization, then reduces spatial size. Filter counts grow as features become more abstract. Global average pooling replaces a large flattened layer, cutting parameters and overfitting. Dropout regularizes the final classifier.

```text
160² RGB → Conv32 → Conv64 → Conv128 → Conv256 → GlobalAvgPool → Dense128 → sigmoid
```


In [5]:
augmentation = tf.keras.Sequential([
    tf.keras.layers.RandomFlip("horizontal"),
    tf.keras.layers.RandomRotation(0.04),
    tf.keras.layers.RandomZoom(0.1),
    tf.keras.layers.RandomContrast(0.1),
], name="augmentation")

model = tf.keras.Sequential([
    tf.keras.layers.Input((IMAGE_SIZE, IMAGE_SIZE, 3)),
    augmentation,
    tf.keras.layers.Conv2D(32, 3, padding="same", activation="relu"),
    tf.keras.layers.BatchNormalization(),
    tf.keras.layers.MaxPool2D(),
    tf.keras.layers.Conv2D(64, 3, padding="same", activation="relu"),
    tf.keras.layers.BatchNormalization(),
    tf.keras.layers.MaxPool2D(),
    tf.keras.layers.Conv2D(128, 3, padding="same", activation="relu"),
    tf.keras.layers.BatchNormalization(),
    tf.keras.layers.MaxPool2D(),
    tf.keras.layers.Conv2D(256, 3, padding="same", activation="relu"),
    tf.keras.layers.BatchNormalization(),
    tf.keras.layers.GlobalAveragePooling2D(),
    tf.keras.layers.Dense(128, activation="relu"),
    tf.keras.layers.Dropout(0.35),
    tf.keras.layers.Dense(1, activation="sigmoid"),
], name=MODEL_ID)


## 6. Compile

Binary cross-entropy matches a two-class probability. Adam adapts each parameter's learning rate. Accuracy gives an intuitive headline and AUC measures ranking quality across every possible threshold.


In [6]:
model.compile(
    optimizer=tf.keras.optimizers.Adam(LEARNING_RATE),
    loss="binary_crossentropy",
    metrics=[tf.keras.metrics.BinaryAccuracy(name="accuracy"), tf.keras.metrics.AUC(name="auc")],
)
model.summary()
print({"parameters": model.count_params(), "artifact": str(ARTIFACT)})


Model: "custom-cnn"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ augmentation (Sequential)       │ (None, 160, 160, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d (Conv2D)                 │ (None, 160, 160, 32)   │           896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization             │ (None, 160, 160, 32)   │           128 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d (MaxPooling2D)    │ (None, 80, 80, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_1 (Conv2D)               │ (None, 80, 80, 64)     │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_1           │ (None, 80, 80, 64)     │           256 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_1 (MaxPooling2D)  │ (None, 40, 40, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_2 (Conv2D)               │ (None, 40, 40, 128)    │        73,856 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_2           │ (None, 40, 40, 128)    │           512 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_2 (MaxPooling2D)  │ (None, 20, 20, 128)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_3 (Conv2D)               │ (None, 20, 20, 256)    │       295,168 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_3           │ (None, 20, 20, 256)    │         1,024 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d        │ (None, 256)            │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 128)            │        32,896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 1)              │           129 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 423,361 (1.61 MB)

 Trainable params: 422,401 (1.61 MB)

 Non-trainable params: 960 (3.75 KB)

{'parameters': 423361, 'artifact': 'C:\\Users\\praloya\\Desktop\\New folder\\projects\\Breathe-AI\\server\\artifacts\\custom-cnn.keras'}


## 7. Train — long-running cell

Checkpointing keeps only the best validation-loss model. Early stopping avoids wasted epochs after learning stalls, and learning-rate reduction makes smaller updates near a good solution. Augmentation is active only during training and improves robustness to small acquisition differences.


In [7]:
callbacks = [
    tf.keras.callbacks.ModelCheckpoint(ARTIFACT, monitor="val_loss", save_best_only=True),
    tf.keras.callbacks.EarlyStopping(monitor="val_loss", patience=7, restore_best_weights=True),
    tf.keras.callbacks.ReduceLROnPlateau(monitor="val_loss", patience=3, factor=0.3, min_lr=1e-7),
]

history = model.fit(
    train_data,
    validation_data=validation_data,
    epochs=EPOCHS,
    class_weight=class_weight,
    callbacks=callbacks,
)
print({
    "epochs_completed": len(history.history["loss"]),
    "best_validation_loss": min(history.history["val_loss"]),
    "best_validation_accuracy": max(history.history["val_accuracy"]),
})


Epoch 1/30


c:\Users\praloya\Desktop\New folder\projects\Breathe-AI\.venv\Lib\site-packages\keras\src\trainers\epoch_iterator.py:74: UserWarning: `shuffle=True` was passed, but will be ignored since the data `x` was provided as a tf.data.Dataset. The Dataset is expected to already be shuffled (via `.shuffle(buffer_size)`).
  self.data_adapter = data_adapters.get_data_adapter(


131/131 ━━━━━━━━━━━━━━━━━━━━ 56s 405ms/step - accuracy: 0.8765 - auc: 0.9509 - loss: 0.2768 - val_accuracy: 0.7421 - val_auc: 0.5000 - val_loss: 4.4041 - learning_rate: 0.0010
Epoch 2/30
131/131 ━━━━━━━━━━━━━━━━━━━━ 54s 409ms/step - accuracy: 0.9087 - auc: 0.9672 - loss: 0.2258 - val_accuracy: 0.7421 - val_auc: 0.5000 - val_loss: 4.2735 - learning_rate: 0.0010
Epoch 3/30
131/131 ━━━━━━━━━━━━━━━━━━━━ 59s 448ms/step - accuracy: 0.9276 - auc: 0.9787 - loss: 0.1824 - val_accuracy: 0.7421 - val_auc: 0.5019 - val_loss: 2.1838 - learning_rate: 0.0010
Epoch 4/30
131/131 ━━━━━━━━━━━━━━━━━━━━ 62s 470ms/step - accuracy: 0.9379 - auc: 0.9821 - loss: 0.1651 - val_accuracy: 0.7517 - val_auc: 0.8995 - val_loss: 0.9509 - learning_rate: 0.0010
Epoch 5/30
131/131 ━━━━━━━━━━━━━━━━━━━━ 62s 469ms/step - accuracy: 0.9395 - auc: 0.9861 - loss: 0.1483 - val_accuracy: 0.6581 - val_auc: 0.9709 - val_loss: 0.9304 - learning_rate: 0.0010
Epoch 6/30
131/131 ━━━━━━━━━━━━━━━━━━━━ 62s 469ms/step - accuracy: 0.9486 - 

## 8. Evaluate on untouched images

Accuracy alone can hide class imbalance, so this cell also computes precision, recall/sensitivity, specificity, F1, AUC, log loss and the full confusion matrix. Threshold 0.5 is recorded so future comparisons are reproducible.


In [8]:
model = tf.keras.models.load_model(ARTIFACT, compile=False)
probabilities = np.asarray(model.predict(test_data, verbose=1)).reshape(-1)
actual = np.concatenate([labels.numpy() for _, labels in test_data]).astype(int)
predicted = (probabilities >= 0.5).astype(int)

tn = int(np.sum((actual == 0) & (predicted == 0)))
fp = int(np.sum((actual == 0) & (predicted == 1)))
fn = int(np.sum((actual == 1) & (predicted == 0)))
tp = int(np.sum((actual == 1) & (predicted == 1)))
divide = lambda numerator, denominator: float(numerator / denominator) if denominator else 0.0
precision = divide(tp, tp + fp)
recall = divide(tp, tp + fn)
clipped = np.clip(probabilities, 1e-7, 1 - 1e-7)

metrics = {
    "status": "verified_local_test_split",
    "accuracy": divide(tp + tn, len(actual)),
    "loss": float(-np.mean(actual * np.log(clipped) + (1 - actual) * np.log(1 - clipped))),
    "precision": precision,
    "recall": recall,
    "f1": divide(2 * precision * recall, precision + recall),
    "specificity": divide(tn, tn + fp),
    "auc": float(tf.keras.metrics.AUC()(actual, probabilities).numpy()),
    "confusion_matrix": {"tn": tn, "fp": fp, "fn": fn, "tp": tp},
    "samples": len(actual),
    "threshold": 0.5,
}
metrics


20/20 ━━━━━━━━━━━━━━━━━━━━ 3s 118ms/step


{'status': 'verified_local_test_split',
 'accuracy': 0.7067307692307693,
 'loss': 1.0590556439408358,
 'precision': 0.681260945709282,
 'recall': 0.9974358974358974,
 'f1': 0.809573361082206,
 'specificity': 0.2222222222222222,
 'auc': 0.8941376209259033,
 'confusion_matrix': {'tn': 52, 'fp': 182, 'fn': 1, 'tp': 389},
 'samples': 624,
 'threshold': 0.5}

## 9. Save and register for inference

The best model already exists at the artifact path. This cell hashes it and writes the verified metrics and version into the shared catalog. The API verifies this hash before loading the model.


In [9]:
def sha256(path):
    digest = hashlib.sha256()
    with path.open("rb") as stream:
        for chunk in iter(lambda: stream.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()

digest = sha256(ARTIFACT)
catalog = json.loads(CATALOG.read_text(encoding="utf-8"))
entry = next(item for item in catalog["models"] if item["id"] == MODEL_ID)
entry["parameters"] = int(model.count_params())
entry["metrics"] = metrics
entry["artifact"].update({
    "version": digest[:12],
    "sha256": digest,
    "status": "trained_and_checksum_verified",
    "trained_at": datetime.now(timezone.utc).isoformat(),
})
CATALOG.write_text(json.dumps(catalog, indent=2) + "\n", encoding="utf-8")
print({"saved": str(ARTIFACT), "model_version": digest[:12], "catalog_updated": str(CATALOG)})


{'saved': 'C:\\Users\\praloya\\Desktop\\New folder\\projects\\Breathe-AI\\server\\artifacts\\custom-cnn.keras', 'model_version': '6057f2bf31a2', 'catalog_updated': 'C:\\Users\\praloya\\Desktop\\New folder\\projects\\Breathe-AI\\server\\model_catalog.json'}


## What to do after running

Restart the backend, refresh the web application, choose **Custom CNN**, and upload an image. Keep the generated artifact local; retraining this notebook replaces only this model's artifact and catalog metrics.
